# Thailand Development Intelligence Pipeline & Dashboard
### Assignment: Local LLMs for Country Development Intelligence and Evaluation

This Jupyter Notebook contains a fully integrated pipeline to extract development intelligence from the **Thailand Human Development Report 2003** PDF. It is optimized to run on **Google Colab** using the T4 GPU runtime. It automates:
1. **PDF Parsing & Text Cleaning**: Extracts raw text, cleans headers/footers, and segments it by chapters.
2. **Vector Store & RAG Pipeline**: Chunks text, creates embeddings using local `nomic-embed-text:latest`, and indexes them in an in-memory vector store.
3. **Local LLM Extraction & Summarisation**: Compares three models (`llama3.2`, `qwen2.5:3b`, `phi3:mini`) on hierarchical summarisation, indicator extraction, and strengths/challenges.
4. **Cross-LLM Evaluation Framework**: Automates circular cross-evaluation (consistency, completeness, factual alignment).
5. **Data Visualization**: Generates 7 interactive plots inside the notebook using Plotly.
6. **Streamlit App Integration & Live Hosting**: Generates the `app.py` script and hosts the interactive dashboard directly inside your Google Colab session using Colab's secure port proxy.

---
## Google Colab Execution Instructions

### Step 1: Change Runtime to GPU
- Go to **Runtime** > **Change runtime type** in the top menu.
- Select **T4 GPU** under Hardware Accelerator.
- Click **Save**.

### Step 2: Run Cells Sequentially
- Execute the setup cells to install Ollama, start the Ollama background process, and pull the required models.
- Upload the `thailand2003en.pdf` when prompted by the file upload cell.

### Step 3: Run the Extraction & Live Dashboard
- Run all pipeline cells. The T4 GPU will complete the extraction in a few minutes.
- The Streamlit cell at the end will launch the app in the background and output a secure link. Click the link to open the dashboard directly in your browser!

### Step 4: Download Results for Local Use (Optional)
- The last cell packages results into a zip file (`thailand_intelligence_results.zip`) and triggers a download. You can run it on your local laptop too.

In [1]:
# Install zstd (required for Ollama extraction in Colab) and other python libraries
!apt-get install -y zstd
!pip install pypdf ollama numpy plotly streamlit

# Install Ollama binary in the Google Colab VM
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (11.2 MB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 94.7 MB/s eta 0:00:00
>>> Install

In [2]:
import subprocess
import os
import time

print("Starting Ollama server in background with strict RAM limits...")

# Configure Ollama to prevent out-of-memory (OOM) crashes in Colab:
# - OLLAMA_MAX_LOADED_MODELS=1: Enforces that at most 1 model is loaded in RAM/VRAM at a time
# - OLLAMA_NUM_PARALLEL=1: Disables concurrent request queues which load duplicate models
# - OLLAMA_KEEP_ALIVE=0s: Forces immediate model unloading after finishing a request
env = os.environ.copy()
env["OLLAMA_MAX_LOADED_MODELS"] = "1"
env["OLLAMA_NUM_PARALLEL"] = "1"
env["OLLAMA_KEEP_ALIVE"] = "0s"

subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # Wait for server to start
print("Ollama server started successfully with RAM optimizations!")

Starting Ollama server in background with strict RAM limits...
Ollama server started successfully with RAM optimizations!


In [3]:
print("Pulling models... This will take a moment depending on network speeds.")
!ollama pull llama3.2:latest
!ollama pull qwen2.5:3b
!ollama pull phi3:mini
!ollama pull nomic-embed-text:latest
print("All models pulled successfully!")

Pulling models... This will take a moment depending on network speeds.




All models pulled successfully!


In [4]:
from google.colab import files
import os

print("Please upload your 'thailand2003en.pdf' file.")
uploaded = files.upload()

pdf_filename = "thailand2003en.pdf"
if pdf_filename in uploaded:
    print(f"File {pdf_filename} uploaded successfully!")
else:
    # Find any uploaded PDF and rename it if needed
    for k in uploaded.keys():
        if k.endswith(".pdf"):
            os.rename(k, pdf_filename)
            print(f"Renamed uploaded file {k} to {pdf_filename}")
            break

Please upload your 'thailand2003en.pdf' file.


Saving thailand2003en.pdf to thailand2003en.pdf
File thailand2003en.pdf uploaded successfully!


In [5]:
import re
import json
import pypdf
import numpy as np
import ollama
import plotly.express as px
import plotly.graph_objects as go

PDF_PATH = "thailand2003en.pdf"

# Chapter page boundaries (1-indexed based on PDF pages)
CHAPTER_RANGES = {
    "Overview": (12, 20),
    "Chapter 1": (23, 46),
    "Chapter 2": (47, 66),
    "Chapter 3": (67, 82),
    "Chapter 4": (83, 100),
    "Chapter 5": (103, 136)
}

def clean_text(text):
    # Remove common headers/footers
    text = re.sub(r'THAILAND HUMAN DEVELOPMENT REPORT 2003\s*\d*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\d*\s*THAILAND HUMAN DEVELOPMENT REPORT 2003', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\d*\s*HUMAN DEVELOPMENT INDICES', '', text, flags=re.IGNORECASE)
    text = re.sub(r'COMMUNITY E MPOWERMENT AND\s*HUMAN D EVELOPMENT\s*PART I', '', text, flags=re.IGNORECASE)
    text = re.sub(r'MEASURING H UMAN D EVELOPMENT\s*PART II', '', text, flags=re.IGNORECASE)

    # Resolve end-of-line hyphenation
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    text = re.sub(r'(\w+)-\s*\n(\w+)', r'\1\2', text)

    # Clean extra whitespaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

print("Cleaning and segmentation helper functions defined!")

Cleaning and segmentation helper functions defined!


In [6]:
class SimpleVectorDB:
    def __init__(self):
        self.documents = []
        self.embeddings = []
        self.metadata = []

    def add_document(self, text, meta):
        clean_d = clean_text(text)
        if not clean_d:
            return
        try:
            res = ollama.embeddings(model="nomic-embed-text:latest", prompt=clean_d, keep_alive="0s")
            emb = res.get("embedding")
            if emb:
                self.documents.append(clean_d)
                self.embeddings.append(emb)
                self.metadata.append(meta)
        except Exception as e:
            print(f"Error embedding page {meta.get('page')}: {e}")

    def query(self, query_text, top_k=3):
        try:
            res = ollama.embeddings(model="nomic-embed-text:latest", prompt=query_text, keep_alive="0s")
            q_emb = res.get("embedding")
            if not q_emb or not self.embeddings:
                return []

            q_v = np.array(q_emb)
            similarities = []
            for emb in self.embeddings:
                e_v = np.array(emb)
                denom = np.linalg.norm(q_v) * np.linalg.norm(e_v)
                sim = np.dot(q_v, e_v) / denom if denom > 0 else 0
                similarities.append(sim)

            top_indices = np.argsort(similarities)[::-1][:top_k]
            results = []
            for idx in top_indices:
                results.append({
                    "text": self.documents[idx],
                    "metadata": self.metadata[idx],
                    "score": float(similarities[idx])
                })
            return results
        except Exception as e:
            print(f"Query error: {e}")
            return []

print("In-memory vector store class created!")

In-memory vector store class created!


In [7]:
print("Parsing PDF and indexing in Vector DB...")
reader = pypdf.PdfReader(PDF_PATH)
db = SimpleVectorDB()

# Segment and clean text page-by-page
for idx, page in enumerate(reader.pages):
    page_num = idx + 1
    text = page.extract_text()
    if text.strip():
        chapter = "Other"
        for ch_name, (start, end) in CHAPTER_RANGES.items():
            if start <= page_num <= end:
                chapter = ch_name
                break
        db.add_document(text, {"page": page_num, "chapter": chapter})

print(f"Indexed {len(db.documents)} pages in the vector database.")

# Build chapter text dictionary for summarisation
chapter_texts = {}
for ch_name, (start, end) in CHAPTER_RANGES.items():
    ch_text = ""
    for p_num in range(start, end+1):
        if p_num <= len(reader.pages):
            txt = reader.pages[p_num - 1].extract_text()
            ch_text += clean_text(txt) + " "
    chapter_texts[ch_name] = ch_text.strip()

Parsing PDF and indexing in Vector DB...
Error embedding page 155: the input length exceeds the context length (status code: 500)
Error embedding page 156: the input length exceeds the context length (status code: 500)
Indexed 167 pages in the vector database.


In [8]:
def parse_json_safely(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            pass
    clean = text.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(clean)
    except:
        return None

print("JSON parser helper configured!")

JSON parser helper configured!


In [9]:
import time

MODELS = {
    "llama": "llama3.2:latest",
    "qwen": "qwen2.5:3b",
    "phi": "phi3:mini"
}

all_results = {}

for model_key, model_name in MODELS.items():
    print(f"\nProcessing model: {model_name}...")
    model_data = {
        "summaries": {},
        "key_results": [],
        "indicators": {},
        "strengths_challenges": {},
        "demographic_trends": [],
        "processing_time_sec": 0.0
    }

    # Track execution duration for model benchmarking
    t_start = time.time()

    # 1. Summaries
    for ch_name, ch_text in chapter_texts.items():
        print(f"  Summarising {ch_name}...")
        words = ch_text.split()
        block_size = 3500
        blocks = [" ".join(words[i:i+block_size]) for i in range(0, len(words), block_size)]

        block_sums = []
        for idx, block in enumerate(blocks):
            prompt = (
                f"Summarize the following text from '{ch_name}' of the Thailand Human Development Report 2003. "
                f"Provide a concise summary focused on core findings and recommendations.\n\n"
                f"Text:\n{block}"
            )
            res = ollama.chat(model=model_name, messages=[{"role": "user", "content": prompt}], keep_alive="0s")
            block_sums.append(res['message']['content'].strip())

        combined = "\n\n".join(block_sums)
        final_prompt = (
            f"Synthesize the following section summaries from '{ch_name}' of the Thailand Human Development Report 2003 "
            f"into a single chapter summary. The summary MUST be strictly under 100 words, fact-focused, and capture main points. "
            f"Do not use phrases like 'This chapter discusses...' or 'In summary...'. Respond with only the summary text.\n\n"
            f"Section Summaries:\n{combined}"
        )
        res = ollama.chat(model=model_name, messages=[{"role": "user", "content": final_prompt}], keep_alive="0s")
        final_sum = res['message']['content'].strip()
        if len(final_sum.split()) > 100:
            final_sum = " ".join(final_sum.split()[:95]) + "..."
        model_data["summaries"][ch_name] = final_sum

    # 2. Key Results
    print("  Extracting Key Results...")
    overview_chunks = db.query("Key results overview and main findings of Thailand Human Development Report 2003", top_k=5)
    context = "\n\n".join([c["text"] for c in overview_chunks])
    prompt = (
        f"Based on the following context from the Thailand Human Development Report 2003, extract 4-5 key overall development results or major findings. "
        f"Provide them as a bulleted list. Each bullet point should be concise and fact-based.\n\n"
        f"Context:\n{context}"
    )
    res = ollama.chat(model=model_name, messages=[{"role": "user", "content": prompt}], keep_alive="0s")
    model_data["key_results"] = [line.strip("- * ") for line in res['message']['content'].strip().split("\n") if line.strip()]

    # 3. Indicators
    print("  Extracting Indicators...")
    ind_chunks = db.query("Table 12 Table 14 national HDI score GDI score life expectancy mean years of schooling GDP per capita population of Thailand", top_k=5)
    context = "\n\n".join([c["text"] for c in ind_chunks])
    prompt = (
        f"Extract the following core national human development indicators for Thailand in 2001-2003 from the provided context. "
        f"If a specific indicator is not found or is subnational, use the national average/Kingdom value. If a value is missing, return null.\n"
        f"Indicators to extract:\n"
        f"1. HDI value (Kingdom overall)\n"
        f"2. HDI rank (if mentioned)\n"
        f"3. Life expectancy at birth (total years)\n"
        f"4. Mean years of schooling (overall)\n"
        f"5. GDP/GPP per capita (annual Baht)\n"
        f"6. Population of Thailand\n\n"
        f"Format your response strictly as a JSON object with the following keys and values (no markdown formatting blocks, return only raw JSON):\n"
        f"{{\n"
        f"  \"hdi_value\": float,\n"
        f"  \"hdi_rank\": string or null,\n"
        f"  \"life_expectancy_total\": float,\n"
        f"  \"mean_years_of_schooling\": float,\n"
        f"  \"gdp_per_capita_baht\": float,\n"
        f"  \"population\": string\n"
        f"}}\n\n"
        f"Context:\n{context}"
    )
    res = ollama.chat(model=model_name, messages=[{"role": "user", "content": prompt}], keep_alive="0s")
    model_data["indicators"] = parse_json_safely(res['message']['content']) or {"error": "parsing_error", "raw": res['message']['content']}

    # 4. Strengths & Challenges
    print("  Extracting Strengths & Challenges...")
    emp_chunks = db.query("community empowerment strengths challenges achievements problems issues Thailand", top_k=5)
    context = "\n\n".join([c["text"] for c in emp_chunks])
    prompt = (
        f"Based on the following context, identify 5 to 8 key development strengths of Thailand and 5 to 8 key development challenges. "
        f"Format the output as a JSON object (no markdown formatting blocks, return only raw JSON):\n"
        f"{{\n"
        f"  \"strengths\": [list of strings],\n"
        f"  \"challenges\": [list of strings]\n"
        f"}}\n\n"
        f"Ensure each item is a concise, fact-based statement.\n\n"
        f"Context:\n{context}"
    )
    res = ollama.chat(model=model_name, messages=[{"role": "user", "content": prompt}], keep_alive="0s")
    model_data["strengths_challenges"] = parse_json_safely(res['message']['content']) or {"error": "parsing_error", "raw": res['message']['content']}

    # 5. Demographic Trends
    print("  Extracting Demographic Trends...")
    trend_chunks = db.query("demographic trends over time poverty rate income 1998 2000 table 5 Thailand", top_k=4)
    context = "\n\n".join([c["text"] for c in trend_chunks])
    prompt = (
        f"Identify quantities showing demographic or economic trends over time from the provided context (e.g. poverty rates or household income change). "
        f"Provide a brief list (2-3 items) of trends with their years and values.\n\n"
        f"Context:\n{context}"
    )
    res = ollama.chat(model=model_name, messages=[{"role": "user", "content": prompt}], keep_alive="0s")
    model_data["demographic_trends"] = [line.strip("- * ") for line in res['message']['content'].strip().split("\n") if line.strip()]

    model_data["processing_time_sec"] = float(time.time() - t_start)
    all_results[model_key] = model_data
    print(f"Finished processing for {model_name} in {model_data['processing_time_sec']:.2f}s!")


Processing model: llama3.2:latest...
  Summarising Overview...
  Summarising Chapter 1...
  Summarising Chapter 2...
  Summarising Chapter 3...
  Summarising Chapter 4...
  Summarising Chapter 5...
  Extracting Key Results...
  Extracting Indicators...
  Extracting Strengths & Challenges...
  Extracting Demographic Trends...
Finished processing for llama3.2:latest in 443.75s!

Processing model: qwen2.5:3b...
  Summarising Overview...
  Summarising Chapter 1...
  Summarising Chapter 2...
  Summarising Chapter 3...
  Summarising Chapter 4...
  Summarising Chapter 5...
  Extracting Key Results...
  Extracting Indicators...
  Extracting Strengths & Challenges...
  Extracting Demographic Trends...
Finished processing for qwen2.5:3b in 444.51s!

Processing model: phi3:mini...
  Summarising Overview...
  Summarising Chapter 1...
  Summarising Chapter 2...
  Summarising Chapter 3...
  Summarising Chapter 4...
  Summarising Chapter 5...
  Extracting Key Results...
  Extracting Indicators...
  

In [10]:
print("Running Cross-LLM Quality Evaluations...")
evaluations = {}

eval_triplets = [
    ("llama", "qwen", MODELS["llama"], MODELS["qwen"]),
    ("qwen", "phi", MODELS["qwen"], MODELS["phi"]),
    ("phi", "llama", MODELS["phi"], MODELS["llama"])
]

for extracted_key, evaluator_key, ext_name, eval_name in eval_triplets:
    print(f"Evaluating {ext_name} extraction using {eval_name}...")

    test_chunks = db.query("Table 12 Table 14 national HDI score GDI score life expectancy mean years of schooling GDP per capita population of Thailand", top_k=3)
    source_text = "\n\n".join([c["text"] for c in test_chunks])

    extracted_data = json.dumps({
        "summaries": all_results[extracted_key]["summaries"],
        "indicators": all_results[extracted_key]["indicators"]
    }, indent=2)

    prompt = (
        f"You are a quality assessment agent. You are evaluating the quality of human development indicators and summaries extracted by another model from the Thailand Human Development Report 2003.\n\n"
        f"Here is the source text retrieved from the report:\n"
        f"{source_text}\n\n"
        f"Here is the extracted data and summaries:\n"
        f"{extracted_data}\n\n"
        f"Assess the extracted output for:\n"
        f"1. Consistency: Is the extracted data consistent with the source text? (Score 1-10)\n"
        f"2. Completeness: Were all required fields extracted correctly? (Score 1-10)\n"
        f"3. Factual Alignment: Are there any factual errors or hallucinations? (Score 1-10)\n\n"
        f"Provide a final summary evaluation in a JSON object (no markdown formatting blocks, return only raw JSON):\n"
        f"{{\n"
        f"  \"consistency_score\": int,\n"
        f"  \"completeness_score\": int,\n"
        f"  \"factual_alignment_score\": int,\n"
        f"  \"comments\": \"concise comments on stability, accuracy, and richness of the extraction\"\n"
        f"}}\n"
    )

    res = ollama.chat(model=eval_name, messages=[{"role": "user", "content": prompt}], keep_alive="0s")
    evaluations[f"{extracted_key}_by_{evaluator_key}"] = parse_json_safely(res['message']['content']) or {"error": "parsing_error", "raw": res['message']['content']}

print("Evaluations complete!")
print(json.dumps(evaluations, indent=2))

Running Cross-LLM Quality Evaluations...
Evaluating llama3.2:latest extraction using qwen2.5:3b...
Evaluating qwen2.5:3b extraction using phi3:mini...
Evaluating phi3:mini extraction using llama3.2:latest...
Evaluations complete!
{
  "llama_by_qwen": {
    "consistency_score": 8,
    "completeness_score": 9,
    "factual_alignment_score": 10,
    "comments": "The extracted data shows good consistency with the source text. All required fields were correctly extracted, and there are no factual errors or hallucinations. The completeness score is high due to only one missing field: 'GDP per capita'. The inconsistency score could be slightly lower for this missing field."
  },
  "qwen_by_phi": {
    "consistency_score": 9,
    "completeness_score": 8,
    "factual_alignment_score": 10,
    "comments": "The extracted data is largely consistent with the source text and factually aligned. It accurately represents key findings from each chapter of 'Community Empowerment and Human Development.' 

In [11]:
print("Saving extraction and evaluation results as JSON...")
for m_key, m_data in all_results.items():
    with open(f"{m_key}_results.json", "w") as f:
        json.dump(m_data, f, indent=2)

with open("evaluation_results.json", "w") as f:
        json.dump(evaluations, f, indent=2)
print("Results successfully saved!")

Saving extraction and evaluation results as JSON...
Results successfully saved!


In [12]:
import os
import plotly.express as px
import plotly.graph_objects as go

print("=== Render and Save Visualisations ===")

# Create plots output directory
os.makedirs("plots", exist_ok=True)

# Helper to write both HTML and PNG formats
def save_plotly_fig(fig, name):
    html_path = f"plots/{name}.html"
    png_path = f"plots/{name}.png"

    # Save interactive HTML
    fig.write_html(html_path)
    print(f"  Saved: {html_path}")

    # Save static PNG (requires kaleido)
    try:
        fig.write_image(png_path)
        print(f"  Saved: {png_path}")
    except Exception as e:
        print(f"  Could not export static PNG for {name}: {e}")

# Ground Truth Values for Reference
regions_data = {
    "Region": ["Bangkok Metropolis", "Bangkok Vicinity", "Eastern Region", "Central Region", "Western Region", "Southern Region", "Northern Region", "Northeastern Region"],
    "Income_1998": [25790, 19262, 12178, 11473, 12461, 11368, 9502, 8411],
    "Income_2000": [28392, 19439, 12480, 12641, 14310, 11536, 8651, 7823],
    "Poverty_2000": [0.3, 1.4, 5.2, 6.1, 6.1, 11.0, 12.2, 28.1],
    "Debt_2000": [363674, 216952, 150887, 104265, 124832, 92220, 81550, 121569], # Average Debt in Baht
    "Malnutrition_1997": [4.01, 1.66, 4.80, 3.26, 6.1, 5.8, 10.2, 12.9], # Under 5 malnutrition %
    "Infant_Mortality_2001": [10.0, 12.6, 5.1, 11.0, 8.0, 12.0, 13.0, 12.9], # Per 1000
    "Schooling_Years": [11.9, 10.5, 8.0, 7.8, 7.2, 7.1, 6.9, 6.7],
    "Secondary_Enrolment": [67.3, 56.2, 54.8, 56.1, 50.4, 50.9, 51.0, 46.5]
}

# Plot 1: Regional Income Recovery (1998 vs 2000)
fig3 = go.Figure()
fig3.add_trace(go.Bar(x=regions_data["Region"], y=regions_data["Income_1998"], name="1998 (Baht/month)", marker_color="#5D6D7E"))
fig3.add_trace(go.Bar(x=regions_data["Region"], y=regions_data["Income_2000"], name="2000 (Baht/month)", marker_color="#1F618D"))
fig3.update_layout(title="Household Income Recovery Trend Post 1997 Crisis by Region", barmode="group", template="plotly_white")
fig3.show()
save_plotly_fig(fig3, "regional_income_recovery")

# Plot 2: Advanced Radar Chart (HAI Dimensions)
categories = ['Health', 'Education', 'Employment', 'Income', 'Housing', 'Family & Community', 'Transport & Comm', 'Participation']
fig4 = go.Figure()
fig4.add_trace(go.Scatterpolar(r=[0.7884, 0.5418, 0.6744, 0.8074, 0.8191, 0.5816, 0.8765, 0.2959], theta=categories, fill='toself', name='Bangkok Metropolis (Highest)'))
fig4.add_trace(go.Scatterpolar(r=[0.6889, 0.5075, 0.5885, 0.5100, 0.7040, 0.6573, 0.5984, 0.6758], theta=categories, fill='toself', name='Kingdom (National)'))
fig4.add_trace(go.Scatterpolar(r=[0.6234, 0.4654, 0.5074, 0.3542, 0.5422, 0.7209, 0.4392, 0.6502], theta=categories, fill='toself', name='Northeastern Region (Lowest)'))
fig4.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 1.0])), title="Radar Chart: 8 Dimensions of Human Achievement Index (HAI)", showlegend=True)
fig4.show()
save_plotly_fig(fig4, "radar_multidimensional_hdi")

# Plot 3: Poverty vs Household Debt
fig5 = go.Figure()
fig5.add_trace(go.Bar(x=regions_data["Region"], y=regions_data["Poverty_2000"], name="Poverty Rate (%)", marker_color="#E74C3C", yaxis="y"))
fig5.add_trace(go.Scatter(x=regions_data["Region"], y=regions_data["Debt_2000"], name="Avg Household Debt (Baht)", line=dict(color="#2ECC71", width=3), yaxis="y2"))
fig5.update_layout(
    title="Regional Socioeconomic Disparity: Poverty Rate vs Household Debt (2000)",
    yaxis=dict(title=dict(text="Poverty Rate (%)", font=dict(color="#E74C3C")), tickfont=dict(color="#E74C3C")),
    yaxis2=dict(title=dict(text="Household Debt (Baht)", font=dict(color="#2ECC71")), tickfont=dict(color="#2ECC71"), anchor="x", overlaying="y", side="right"),
    template="plotly_white"
)
fig5.show()
save_plotly_fig(fig5, "poverty_vs_debt")

# Plot 4: Child Health Disparities
fig6 = go.Figure()
fig6.add_trace(go.Bar(x=regions_data["Region"], y=regions_data["Malnutrition_1997"], name="Under-5 Malnutrition Rate (1997 %)", marker_color="#F39C12"))
fig6.add_trace(go.Bar(x=regions_data["Region"], y=regions_data["Infant_Mortality_2001"], name="Infant Mortality Rate (2001 per 1000)", marker_color="#9B59B6"))
fig6.update_layout(title="Child Health Disparities: Malnutrition vs Infant Mortality by Region", barmode="group", template="plotly_white")
fig6.show()
save_plotly_fig(fig6, "child_health_disparities")

# Plot 5: Education Development Metrics
fig7 = go.Figure()
fig7.add_trace(go.Bar(x=regions_data["Region"], y=regions_data["Schooling_Years"], name="Mean Years of Schooling", marker_color="#16A085"))
fig7.add_trace(go.Bar(x=regions_data["Region"], y=regions_data["Secondary_Enrolment"], name="Secondary Enrolment Rate (%)", marker_color="#F1C40F"))
fig7.update_layout(title="Regional Education Development Metrics", barmode="group", template="plotly_white")
fig7.show()
save_plotly_fig(fig7, "education_metrics")

print("\n=== Model Benchmark Visualisations ===")
try:
    models_list = ["llama", "qwen", "phi"]
    model_labels = ["Llama 3.2 3B", "Qwen 2.5 3B", "Phi-3 Mini 3.8B"]

    # Benchmark Plot 6: Model Verbosity (Summary lengths)
    chapters = ["Overview", "Chapter 1", "Chapter 2", "Chapter 3", "Chapter 4", "Chapter 5"]
    fig_verb = go.Figure()
    for m_idx, m_key in enumerate(models_list):
        w_counts = [len(all_results[m_key]["summaries"].get(ch, "").split()) for ch in chapters]
        fig_verb.add_trace(go.Bar(x=chapters, y=w_counts, name=model_labels[m_idx]))
    fig_verb.update_layout(title="Model Comparison: Chapter Summary Lengths (Word Count)", barmode="group", template="plotly_white", yaxis_title="Words")
    fig_verb.show()
    save_plotly_fig(fig_verb, "model_comparison_verbosity")

    # Benchmark Plot 7: Quality Validation Scores
    eval_metrics = ["Consistency", "Completeness", "Factual Alignment"]
    fig_e = go.Figure()
    llama_scores = [evaluations.get("llama_by_qwen", {}).get("consistency_score", 0), evaluations.get("llama_by_qwen", {}).get("completeness_score", 0), evaluations.get("llama_by_qwen", {}).get("factual_alignment_score", 0)]
    qwen_scores = [evaluations.get("qwen_by_phi", {}).get("consistency_score", 0), evaluations.get("qwen_by_phi", {}).get("completeness_score", 0), evaluations.get("qwen_by_phi", {}).get("factual_alignment_score", 0)]
    phi_scores = [evaluations.get("phi_by_llama", {}).get("consistency_score", 0), evaluations.get("phi_by_llama", {}).get("completeness_score", 0), evaluations.get("phi_by_llama", {}).get("factual_alignment_score", 0)]

    fig_e.add_trace(go.Bar(x=eval_metrics, y=llama_scores, name="Llama 3.2 (by Qwen)", marker_color="#3498DB"))
    fig_e.add_trace(go.Bar(x=eval_metrics, y=qwen_scores, name="Qwen 2.5 (by Phi)", marker_color="#E67E22"))
    fig_e.add_trace(go.Bar(x=eval_metrics, y=phi_scores, name="Phi-3 (by Llama)", marker_color="#2ECC71"))
    fig_e.update_layout(title="Model Comparison: Cross-LLM Validation Quality (1-10)", barmode="group", template="plotly_white", yaxis_range=[0, 10], yaxis_title="Score")
    fig_e.show()
    save_plotly_fig(fig_e, "model_comparison_quality")

    # Benchmark Plot 8: Execution speed
    times = [all_results[m_key].get("processing_time_sec", 0) for m_key in models_list]
    fig_t = go.Figure()
    fig_t.add_trace(go.Bar(x=model_labels, y=times, marker_color=["#3498DB", "#E67E22", "#2ECC71"]))
    fig_t.update_layout(title="Model Comparison: Extraction Processing Time (Seconds)", template="plotly_white", yaxis_title="Seconds")
    fig_t.show()
    save_plotly_fig(fig_t, "model_comparison_efficiency")
except Exception as err:
    print(f"Error rendering model comparisons: {err}")

=== Render and Save Visualisations ===


  Saved: plots/regional_income_recovery.html
  Could not export static PNG for regional_income_recovery: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



  Saved: plots/radar_multidimensional_hdi.html
  Could not export static PNG for radar_multidimensional_hdi: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



  Saved: plots/poverty_vs_debt.html
  Could not export static PNG for poverty_vs_debt: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



  Saved: plots/child_health_disparities.html
  Could not export static PNG for child_health_disparities: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



  Saved: plots/education_metrics.html
  Could not export static PNG for education_metrics: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido


=== Model Benchmark Visualisations ===


  Saved: plots/model_comparison_verbosity.html
  Could not export static PNG for model_comparison_verbosity: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



  Saved: plots/model_comparison_quality.html
  Could not export static PNG for model_comparison_quality: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



  Saved: plots/model_comparison_efficiency.html
  Could not export static PNG for model_comparison_efficiency: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



In [13]:
streamlit_code = """import streamlit as st
import json
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(
    page_title="Thailand Development Intelligence Dashboard",
    page_icon="🇹🇭",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom premium styling
st.markdown(
    \"\"\"
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;600;700&display=swap');
    html, body, [class*=\"css\"] { font-family: 'Outfit', sans-serif; }
    .metric-card {
        background: rgba(255, 255, 255, 0.05);
        border-radius: 12px;
        padding: 20px;
        border: 1px solid rgba(255, 255, 255, 0.1);
        box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
    }
    .section-title { font-weight: 700; color: #1F618D; margin-top: 20px; }
    </style>
    \"\"\",
    unsafe_allow_html=True
)

# Load Data Functions
@st.cache_data
def load_results():
    models = [\"llama\", \"qwen\", \"phi\"]
    results = {}
    for m in models:
        path = f\"{m}_results.json\"
        if os.path.exists(path):
            with open(path, \"r\") as f:
                results[m] = json.load(f)
        else:
            # Mock fallback if loaded standalone
            results[m] = {
                \"summaries\": {\"Overview\": \"Summary of overview...\", \"Chapter 1\": \"Summary of Ch 1...\"},
                \"key_results\": [\"Key result 1\", \"Key result 2\"],
                \"indicators\": {\"hdi_value\": 0.726, \"life_expectancy_total\": 72.2, \"mean_years_of_schooling\": 7.3, \"gdp_per_capita_baht\": 74675, \"population\": \"62.1 million\"},
                \"strengths_challenges\": {\"strengths\": [\"Strong health system\"], \"challenges\": [\"Poverty in Northeast\"]},
                \"demographic_trends\": [\"Trend 1\"],
                \"processing_time_sec\": 45.2
            }
    eval_path = \"evaluation_results.json\"
    evaluations = {}
    if os.path.exists(eval_path):
        with open(eval_path, \"r\") as f:
            evaluations = json.load(f)
    else:
        evaluations = {
            \"llama_by_qwen\": {\"consistency_score\": 9, \"completeness_score\": 8, \"factual_alignment_score\": 9, \"comments\": \"High quality\"},
            \"qwen_by_phi\": {\"consistency_score\": 8, \"completeness_score\": 9, \"factual_alignment_score\": 8, \"comments\": \"Rich output\"},
            \"phi_by_llama\": {\"consistency_score\": 8, \"completeness_score\": 7, \"factual_alignment_score\": 9, \"comments\": \"Accurate but brief\"}
        }
    return results, evaluations

results, evaluations = load_results()

st.title(\"🇹🇭 Thailand Development Intelligence Dashboard\")
st.markdown(\"**Interactive evaluation dashboard of subnational human achievement and cross-LLM performance analysis.**\")

# Sidebar configuration
st.sidebar.title(\"Dashboard Controls\")
selected_model = st.sidebar.selectbox(\"Select Primary LLM for Overview\", [\"Llama 3.2 3B\", \"Qwen 2.5 3B\", \"Phi-3 Mini 3.8B\"])
model_map = {\"Llama 3.2 3B\": \"llama\", \"Qwen 2.5 3B\": \"qwen\", \"Phi-3 Mini 3.8B\": \"phi\"}
m_key = model_map[selected_model]

tabs = st.tabs([\"Executive Summary\", \"Chapter Summaries\", \"Interactive Data Exploration\", \"Advanced Radar Analysis\", \"Cross-LLM Behavior Analysis\"])

# Tab 1: Executive Summary
with tabs[0]:
    st.header(\"National Overview & Key Insights\")
    col1, col2, col3, col4 = st.columns(4)
    with col1:
        st.metric(\"National HDI Value\", \"0.7263\")
    with col2:
        st.metric(\"Life Expectancy at Birth\", \"72.2 Years\")
    with col3:
        st.metric(\"Mean Years of Schooling\", \"7.3 Years\")
    with col4:
        st.metric(\"GDP Per Capita (Baht)\", \"74,675\")

    st.markdown(\"---\")
    col_left, col_right = st.columns(2)
    with col_left:
        st.subheader(\"Key Development Strengths\")
        strengths = results[m_key][\"strengths_challenges\"].get(\"strengths\", [])
        for s in strengths:
            st.markdown(f\"- {s}\")
    with col_right:
        st.subheader(\"Key Development Challenges\")
        challenges = results[m_key][\"strengths_challenges\"].get(\"challenges\", [])
        for c in challenges:
            st.markdown(f\"- {c}\")

# Tab 2: Chapter Summaries
with tabs[1]:
    st.header(\"Chapter-by-Chapter Summaries\")
    chapter = st.selectbox(\"Select Chapter\", list(results[m_key][\"summaries\"].keys()))
    st.info(results[m_key][\"summaries\"][chapter])

# Tab 3: Interactive Data Exploration
with tabs[2]:
    st.header(\"Regional Disparity Exploration\")

    # Data
    regions_data = {
        \"Region\": [\"Bangkok Metropolis\", \"Bangkok Vicinity\", \"Eastern Region\", \"Central Region\", \"Western Region\", \"Southern Region\", \"Northern Region\", \"Northeastern Region\"],
        \"Income_1998\": [25790, 19262, 12178, 11473, 12461, 11368, 9502, 8411],
        \"Income_2000\": [28392, 19439, 12480, 12641, 14310, 11536, 8651, 7823],
        \"Poverty_2000\": [0.3, 1.4, 5.2, 6.1, 6.1, 11.0, 12.2, 28.1],
        \"Debt_2000\": [363674, 216952, 150887, 104265, 124832, 92220, 81550, 121569],
        \"Malnutrition_1997\": [4.01, 1.66, 4.80, 3.26, 6.1, 5.8, 10.2, 12.9],
        \"Infant_Mortality_2001\": [10.0, 12.6, 5.1, 11.0, 8.0, 12.0, 13.0, 12.9],
        \"Schooling_Years\": [11.9, 10.5, 8.0, 7.8, 7.2, 7.1, 6.9, 6.7],
        \"Secondary_Enrolment\": [67.3, 56.2, 54.8, 56.1, 50.4, 50.9, 51.0, 46.5]
    }

    col_plot1, col_plot2 = st.columns(2)
    with col_plot1:
        st.subheader(\"Income Trend (1998 vs 2000)\")
        fig3 = go.Figure()
        fig3.add_trace(go.Bar(x=regions_data[\"Region\"], y=regions_data[\"Income_1998\"], name=\"1998\", marker_color=\"#5D6D7E\"))
        fig3.add_trace(go.Bar(x=regions_data[\"Region\"], y=regions_data[\"Income_2000\"], name=\"2000\", marker_color=\"#1F618D\"))
        fig3.update_layout(barmode=\"group\", template=\"plotly_white\", height=400)
        st.plotly_chart(fig3, use_container_width=True)

    with col_plot2:
        st.subheader(\"Poverty Rate vs Household Debt\")
        fig5 = go.Figure()
        fig5.add_trace(go.Bar(x=regions_data[\"Region\"], y=regions_data[\"Poverty_2000\"], name=\"Poverty Rate (%)\", marker_color=\"#E74C3C\", yaxis=\"y\"))
        fig5.add_trace(go.Scatter(x=regions_data[\"Region\"], y=regions_data[\"Debt_2000\"], name=\"Avg Debt (Baht)\", line=dict(color=\"#2ECC71\", width=3), yaxis=\"y2\"))
        fig5.update_layout(
            yaxis=dict(title=dict(text=\"Poverty Rate (%)\", font=dict(color=\"#E74C3C\")), tickfont=dict(color=\"#E74C3C\")),
            yaxis2=dict(title=dict(text=\"Household Debt (Baht)\", font=dict(color=\"#2ECC71\")), tickfont=dict(color=\"#2ECC71\"), anchor=\"x\", overlaying=\"y\", side=\"right\"),
            template=\"plotly_white\", height=400
        )
        st.plotly_chart(fig5, use_container_width=True)

    col_plot3, col_plot4 = st.columns(2)
    with col_plot3:
        st.subheader(\"Child Health Disparities\")
        fig6 = go.Figure()
        fig6.add_trace(go.Bar(x=regions_data[\"Region\"], y=regions_data[\"Malnutrition_1997\"], name=\"Under-5 Malnutrition (%)\", marker_color=\"#F39C12\"))
        fig6.add_trace(go.Bar(x=regions_data[\"Region\"], y=regions_data[\"Infant_Mortality_2001\"], name=\"Infant Mortality (per 1,000)\", marker_color=\"#9B59B6\"))
        fig6.update_layout(barmode=\"group\", template=\"plotly_white\", height=400)
        st.plotly_chart(fig6, use_container_width=True)
    with col_plot4:
        st.subheader(\"Education Development Metrics\")
        fig7 = go.Figure()
        fig7.add_trace(go.Bar(x=regions_data[\"Region\"], y=regions_data[\"Schooling_Years\"], name=\"Mean Years of Schooling\", marker_color=\"#16A085\"))
        fig7.add_trace(go.Bar(x=regions_data[\"Region\"], y=regions_data[\"Secondary_Enrolment\"], name=\"Secondary Enrolment Rate (%)\", marker_color=\"#F1C40F\"))
        fig7.update_layout(barmode=\"group\", template=\"plotly_white\", height=400)
        st.plotly_chart(fig7, use_container_width=True)

# Tab 4: Advanced Radar Analysis
with tabs[3]:
    st.header(\"Subnational Multidimensional Development (HAI Dimensions)\")
    categories = ['Health', 'Education', 'Employment', 'Income', 'Housing', 'Family & Community', 'Transport & Comm', 'Participation']

    # Add selection
    selected_regions = st.multiselect(\"Select Regions to Compare\", [\"Kingdom (National)\", \"Bangkok Metropolis\", \"Northeastern Region\"], default=[\"Kingdom (National)\", \"Bangkok Metropolis\", \"Northeastern Region\"])

    fig4 = go.Figure()
    if \"Bangkok Metropolis\" in selected_regions:
        fig4.add_trace(go.Scatterpolar(r=[0.7884, 0.5418, 0.6744, 0.8074, 0.8191, 0.5816, 0.8765, 0.2959], theta=categories, fill='toself', name='Bangkok Metropolis'))
    if \"Kingdom (National)\" in selected_regions:
        fig4.add_trace(go.Scatterpolar(r=[0.6889, 0.5075, 0.5885, 0.5100, 0.7040, 0.6573, 0.5984, 0.6758], theta=categories, fill='toself', name='Kingdom (National)'))
    if \"Northeastern Region\" in selected_regions:
        fig4.add_trace(go.Scatterpolar(r=[0.6234, 0.4654, 0.5074, 0.3542, 0.5422, 0.7209, 0.4392, 0.6502], theta=categories, fill='toself', name='Northeastern Region'))

    fig4.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 1.0])), height=500)
    st.plotly_chart(fig4, use_container_width=True)

# Tab 5: Cross-LLM Behavior Analysis
with tabs[4]:
    st.header(\"Cross-LLM Behavior & Validation Quality\")

    # Comparison Table of extracted indicators
    comp_data = {
        \"Indicator\": [\"HDI Value\", \"Life Expectancy\", \"Mean Schooling\", \"GDP Per Capita (Baht)\", \"Population\"],
        \"Ground Truth\": [0.7263, 72.2, 7.3, 74675, \"62.1-62.3 Million\"],
        \"Llama 3.2 3B\": [results[\"llama\"][\"indicators\"].get(\"hdi_value\"), results[\"llama\"][\"indicators\"].get(\"life_expectancy_total\"), results[\"llama\"][\"indicators\"].get(\"mean_years_of_schooling\"), results[\"llama\"][\"indicators\"].get(\"gdp_per_capita_baht\"), results[\"llama\"][\"indicators\"].get(\"population\")],
        \"Qwen 2.5 3B\": [results[\"qwen\"][\"indicators\"].get(\"hdi_value\"), results[\"qwen\"][\"indicators\"].get(\"life_expectancy_total\"), results[\"qwen\"][\"indicators\"].get(\"mean_years_of_schooling\"), results[\"qwen\"][\"indicators\"].get(\"gdp_per_capita_baht\"), results[\"qwen\"][\"indicators\"].get(\"population\")],
        \"Phi-3 Mini 3.8B\": [results[\"phi\"][\"indicators\"].get(\"hdi_value\"), results[\"phi\"][\"indicators\"].get(\"life_expectancy_total\"), results[\"phi\"][\"indicators\"].get(\"mean_years_of_schooling\"), results[\"phi\"][\"indicators\"].get(\"gdp_per_capita_baht\"), results[\"phi\"][\"indicators\"].get(\"population\")]
    }

    st.subheader(\"Accuracy Verification: Extracted Indicators vs Ground Truth\")
    # Cast dataframe to str to prevent PyArrow serialization errors from mixed numeric/string columns
    st.table(pd.DataFrame(comp_data).astype(str))

    st.subheader(\"Circular Cross-LLM Evaluation Scores\")
    eval_summary = {
        \"Extraction Evaluated\": [\"Llama 3.2 (by Qwen)\", \"Qwen 2.5 (by Phi)\", \"Phi-3 (by Llama)\"],
        \"Consistency (1-10)\": [evaluations.get(\"llama_by_qwen\", {}).get(\"consistency_score\"), evaluations.get(\"qwen_by_phi\", {}).get(\"consistency_score\"), evaluations.get(\"phi_by_llama\", {}).get(\"consistency_score\")],
        \"Completeness (1-10)\": [evaluations.get(\"llama_by_qwen\", {}).get(\"completeness_score\"), evaluations.get(\"qwen_by_phi\", {}).get(\"completeness_score\"), evaluations.get(\"phi_by_llama\", {}).get(\"completeness_score\")],
        \"Factual Alignment (1-10)\": [evaluations.get(\"llama_by_qwen\", {}).get(\"factual_alignment_score\"), evaluations.get(\"qwen_by_phi\", {}).get(\"factual_alignment_score\"), evaluations.get(\"phi_by_llama\", {}).get(\"factual_alignment_score\")],
        \"Review Comments\": [evaluations.get(\"llama_by_qwen\", {}).get(\"comments\"), evaluations.get(\"qwen_by_phi\", {}).get(\"comments\"), evaluations.get(\"phi_by_llama\", {}).get(\"comments\")]
    }
    st.table(pd.DataFrame(eval_summary).astype(str))

    st.markdown(\"---\")
    st.subheader(\"Visual Model Benchmarks\")
    col_bench1, col_bench2 = st.columns(2)
    with col_bench1:
        st.markdown(\"**Model Summary Lengths (Word Count)**\")
        chapters = [\"Overview\", \"Chapter 1\", \"Chapter 2\", \"Chapter 3\", \"Chapter 4\", \"Chapter 5\"]
        fig_verb = go.Figure()
        for m_key, m_label in [(\"llama\", \"Llama 3.2 3B\"), (\"qwen\", \"Qwen 2.5 3B\"), (\"phi\", \"Phi-3 Mini 3.8B\")]:
            word_counts = [len(results[m_key][\"summaries\"].get(ch, \"\").split()) for ch in chapters]
            fig_verb.add_trace(go.Bar(x=chapters, y=word_counts, name=m_label))
        fig_verb.update_layout(barmode=\"group\", template=\"plotly_white\", height=350, yaxis_title=\"Word Count\")
        st.plotly_chart(fig_verb, use_container_width=True)
    with col_bench2:
        st.markdown(\"**Circular Cross-Evaluation Scores (1-10)**\")
        eval_metrics = [\"Consistency\", \"Completeness\", \"Factual Alignment\"]
        fig_e = go.Figure()
        llama_s = [evaluations.get(\"llama_by_qwen\", {}).get(\"consistency_score\", 0), evaluations.get(\"llama_by_qwen\", {}).get(\"completeness_score\", 0), evaluations.get(\"llama_by_qwen\", {}).get(\"factual_alignment_score\", 0)]
        qwen_s = [evaluations.get(\"qwen_by_phi\", {}).get(\"consistency_score\", 0), evaluations.get(\"qwen_by_phi\", {}).get(\"completeness_score\", 0), evaluations.get(\"qwen_by_phi\", {}).get(\"factual_alignment_score\", 0)]
        phi_s = [evaluations.get(\"phi_by_llama\", {}).get(\"consistency_score\", 0), evaluations.get(\"phi_by_llama\", {}).get(\"completeness_score\", 0), evaluations.get(\"phi_by_llama\", {}).get(\"factual_alignment_score\", 0)]
        fig_e.add_trace(go.Bar(x=eval_metrics, y=llama_s, name=\"Llama 3.2\", marker_color=\"#3498DB\"))
        fig_e.add_trace(go.Bar(x=eval_metrics, y=qwen_s, name=\"Qwen 2.5\", marker_color=\"#E67E22\"))
        fig_e.add_trace(go.Bar(x=eval_metrics, y=phi_s, name=\"Phi-3\", marker_color=\"#2ECC71\"))
        fig_e.update_layout(barmode=\"group\", template=\"plotly_white\", height=350, yaxis_range=[0, 10])
        st.plotly_chart(fig_e, use_container_width=True)

    st.markdown(\"**Model Extraction Pipeline Speed (Seconds)**\")
    times = [results[m_key].get(\"processing_time_sec\", 0.0) for m_key in [\"llama\", \"qwen\", \"phi\"]]
    fig_t = go.Figure()
    fig_t.add_trace(go.Bar(x=[\"Llama 3.2 3B\", \"Qwen 2.5 3B\", \"Phi-3 Mini 3.8B\"], y=times, marker_color=[\"#3498DB\", \"#E67E22\", \"#2ECC71\"]))
    fig_t.update_layout(template=\"plotly_white\", height=280, yaxis_title=\"Seconds\")
    st.plotly_chart(fig_t, use_container_width=True)
"""

with open("app.py", "w") as f:
    f.write(streamlit_code)
print("app.py generated successfully!")

app.py generated successfully!


In [14]:
import urllib.request
import subprocess
import time



import subprocess
import time

print("Starting Streamlit dashboard in background with CORS and XSRF disabled...")

# Start Streamlit with disabled CORS and XSRF check to allow WebSocket connections from Colab proxy origins
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

try:
    from google.colab.output import eval_js
    # Native secure proxy URL mapping for Google Colab port 8501
    proxy_url = eval_js("google.colab.kernel.proxyPort(8501)")
    print("\n=========================================================================")
    print("SUCCESS! Your interactive Streamlit dashboard is running natively.")
    print(f"Click the link below to open and view the dashboard in your browser:\n")
    print(proxy_url)
    print("=========================================================================")
except ImportError:
    print("\nNot running in Google Colab environment.")
    print("To run the dashboard locally, execute this command in your terminal:")
    print("streamlit run app.py")

Starting Streamlit dashboard in background with CORS and XSRF disabled...

SUCCESS! Your interactive Streamlit dashboard is running natively.
Click the link below to open and view the dashboard in your browser:

https://8501-gpu-t4-s-kkb-euw4c0-2pfif9pmqix2p-c.europe-west4-0.prod.colab.dev


In [15]:
import zipfile
import os

files_to_zip = [
    "llama_results.json",
    "qwen_results.json",
    "phi_results.json",
    "evaluation_results.json",
    "app.py"
]

zip_name = "thailand_intelligence_results.zip"
print(f"Zipping results into {zip_name}...")
with zipfile.ZipFile(zip_name, 'w') as zipf:
    # Write base files
    for file in files_to_zip:
        if os.path.exists(file):
            zipf.write(file)
            print(f"  Added: {file}")

    # Write plots folder contents recursively
    if os.path.exists("plots"):
        for root, dirs, files in os.walk("plots"):
            for file in files:
                filepath = os.path.join(root, file)
                # Keep relative path under the zip structure
                zipf.write(filepath, os.path.relpath(filepath, "."))
                print(f"  Added: {filepath}")

print("Packaging complete!")

# Trigger download in Colab
try:
    from google.colab import files
    files.download(zip_name)
    print("Download trigger activated!")
except ImportError:
    print("Not in Colab. Please download the zip manually.")

Zipping results into thailand_intelligence_results.zip...
  Added: llama_results.json
  Added: qwen_results.json
  Added: phi_results.json
  Added: evaluation_results.json
  Added: app.py
  Added: plots/radar_multidimensional_hdi.html
  Added: plots/model_comparison_efficiency.html
  Added: plots/model_comparison_verbosity.html
  Added: plots/child_health_disparities.html
  Added: plots/poverty_vs_debt.html
  Added: plots/regional_income_recovery.html
  Added: plots/education_metrics.html
  Added: plots/model_comparison_quality.html
Packaging complete!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download trigger activated!
